# [16.7] Data Shapley in One Training Run

## Core question

Can exact Data Shapley, sampled permutation Data Shapley, and a one-run gradient-dot proxy all identify the same harmful training example?

## Learning objectives

By the end, you should be able to:

1. Define training-example utility as validation-loss improvement after one step.
2. Enumerate complete training-example coalition tables.
3. Compute exact Data Shapley values for a tiny training run.
4. Approximate Data Shapley with sampled training-example permutations.
5. Compare exact values with one-run gradient-dot scores.
6. Reject random-data and label-shuffled controls.
7. Interpret the CUDA report without claiming production-scale data valuation.

> Difficulty: 4/5  
> Importance: 4/5

<img src="../../instructions/assets/data_shapley_validation_loop.svg" width="760">

The toy problem has three helpful examples and one flipped-label harmful example. Exact values should be `[0.6412, 0.6412, 0.6412, -1.1736]`.

<details><summary>Help - why start this small?</summary>

All 16 coalitions can be enumerated, so the exact harmful-example target is not ambiguous. Larger data-attribution methods should earn trust against this toy oracle before making real-data claims.

</details>


## Setup

Run this once. The tests are deterministic and small; the final CUDA cell can rerun the one-step model-organism path from the solution module.

<details><summary>Expected output</summary>

No printed output. Imports should succeed.

</details>


In [ ]:
from collections.abc import Callable, Mapping
import itertools
import json
import random
import sys
import time
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part7_data_shapley_in_one_training_run"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part7_data_shapley_in_one_training_run.tests as tests

Coalition = frozenset[int]
MAIN = True


## Toy problem

The fourth label is flipped. This is the known harmful example.

<details><summary>Expected output</summary>

No printed output. `toy_data_shapley_problem` should define four training examples and one validation example.

</details>


In [ ]:
DATA_SHAPLEY_LR = 0.5
DATA_SHAPLEY_MC_SAMPLES = 512
DATA_SHAPLEY_RANDOM_CONTROL_SEED = 13
DATA_SHAPLEY_LABEL_SHUFFLE_PERMUTATION = (0, 3, 2, 1)
DATA_SHAPLEY_RUNTIME_REPEATS = 128


def toy_data_shapley_problem() -> tuple[t.Tensor, t.Tensor, t.Tensor, t.Tensor]:
    train_x = t.ones(4, 1, dtype=t.float64)
    train_y = t.tensor([1.0, 1.0, 1.0, -1.0], dtype=t.float64)
    val_x = t.ones(1, 1, dtype=t.float64)
    val_y = t.ones(1, dtype=t.float64)
    return train_x, train_y, val_x, val_y


## Exercise 1 - one-step utility

Implement validation-loss improvement after one gradient step on a coalition.

<details><summary>Expected output</summary>

```text
All tests in `test_one_step_linear_utility_toy_oracle` passed!
```

</details>

<details><summary>What you should see</summary>

```text
v(empty) = 0.0
v({helpful}) = 1.0
v({flipped}) = -3.0
v(all) = 0.75
```

</details>

<details><summary>Common bugs</summary>

- Using training loss instead of validation-loss improvement.
- Updating on the validation example.
- Letting the empty coalition train.

</details>

<details><summary>Solution</summary>

Compute the baseline validation loss at zero weight, take one analytic gradient step on the selected training examples, and subtract the updated validation loss.

</details>


In [ ]:
def one_step_linear_utility(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
    coalition: Coalition,
    *,
    learning_rate: float = DATA_SHAPLEY_LR,
) -> float:
    """Return validation-loss improvement after one step on `coalition`."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_one_step_linear_utility_toy_oracle(one_step_linear_utility)


## Exercise 2 - exact Data Shapley

Enumerate every training-example coalition and compute exact weighted marginal contributions.

<details><summary>Expected output</summary>

```text
All tests in `test_data_coalition_values_complete_table` passed!
All tests in `test_exact_data_shapley_values_matches_hand_checked_result` passed!
```

</details>

<details><summary>What you should see</summary>

```text
exact_values = [0.6412, 0.6412, 0.6412, -1.1736]
harmful_index = 3
```

</details>

<details><summary>Common bugs</summary>

- Treating features as players instead of training examples.
- Missing the empty or full coalition.
- Forgetting Shapley's factorial weights.

</details>

<details><summary>Solution</summary>

Evaluate all `2**4` coalitions with your utility function, then apply exact Shapley's weighted marginal formula.

</details>


In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    """Return every training-example coalition."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def data_coalition_values(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
    *,
    learning_rate: float = DATA_SHAPLEY_LR,
) -> dict[Coalition, float]:
    """Evaluate one-step utility on every training-example coalition."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    """Compute exact Shapley values from a complete coalition table."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def exact_data_shapley_values(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
    *,
    learning_rate: float = DATA_SHAPLEY_LR,
) -> t.Tensor:
    """Compute exact Data Shapley values for the one-step problem."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_data_coalition_values_complete_table(data_coalition_values)
    tests.test_exact_data_shapley_values_matches_hand_checked_result(
        exact_data_shapley_values
    )


## Exercise 3 - sampled permutation Data Shapley

Estimate Data Shapley by averaging marginal utility over sampled training-example orderings.

<details><summary>Expected output</summary>

```text
All tests in `test_sampled_permutation_data_shapley_approximates_exact` passed!
```

</details>

<details><summary>Common bugs</summary>

- Sampling arbitrary coalitions instead of permutations.
- Forgetting to update the running coalition.
- Using too few samples and losing the harmful-example ranking.

</details>

<details><summary>Solution</summary>

For each sampled ordering, walk from empty coalition to full coalition and add each marginal utility jump to the entering example.

</details>


In [ ]:
def sampled_permutation_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    num_samples: int,
    seed: int = 0,
) -> t.Tensor:
    """Estimate Shapley values by sampling random player orderings."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_sampled_permutation_data_shapley_approximates_exact(
        sampled_permutation_shapley_values
    )


## Exercise 4 - in-run first-order scores

Compute per-example train gradients and the validation gradient at initialization, then take gradient-dot scores.

<details><summary>Expected output</summary>

```text
All tests in `test_in_run_first_order_scores_toy_oracle` passed!
```

</details>

<details><summary>What you should see</summary>

```text
gradient_scores = [4.0, 4.0, 4.0, -4.0]
```

</details>

<details><summary>Solution</summary>

At zero weight, compute the validation gradient and the per-example training gradients, then return the dot products.

</details>


In [ ]:
def in_run_first_order_data_scores(
    train_x: t.Tensor,
    train_y: t.Tensor,
    val_x: t.Tensor,
    val_y: t.Tensor,
) -> t.Tensor:
    """Compute per-example gradient-dot scores from initialization."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_in_run_first_order_scores_toy_oracle(in_run_first_order_data_scores)


## Exercise 5 - reports and controls

Package exact, sampled, in-run, random-data, and label-shuffled reports.

<details><summary>Expected output</summary>

```text
All tests in `test_exact_data_shapley_smoke_test` passed!
All tests in `test_monte_carlo_data_shapley_smoke_test` passed!
All tests in `test_in_run_data_shapley_smoke_test` passed!
All tests in `test_random_data_attribution_failure_smoke_test` passed!
All tests in `test_label_shuffled_attribution_failure_smoke_test` passed!
```

</details>

<details><summary>Help - why must controls fail?</summary>

If random data or shuffled labels preserve the planted signal, the method is measuring the wrong thing. A failed control is required before the positive toy result is meaningful.

</details>

<details><summary>Solution</summary>

Use exact values as the signal, then verify random-data and label-shuffled variants do not recover the planted harmful index or correlation.

</details>


In [ ]:
def exact_data_shapley_smoke_test() -> dict:
    """Return exact Data Shapley metrics for the toy problem."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def monte_carlo_data_shapley_smoke_test() -> dict:
    """Return sampled permutation Data Shapley metrics."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def in_run_data_shapley_smoke_test() -> dict:
    """Return exact-vs-in-run proxy metrics."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def random_data_attribution_failure_smoke_test() -> dict:
    """Return the deterministic random-data negative control."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def label_shuffled_attribution_failure_smoke_test() -> dict:
    """Return the deterministic label-shuffle negative control."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_exact_data_shapley_smoke_test(exact_data_shapley_smoke_test)
    tests.test_monte_carlo_data_shapley_smoke_test(monte_carlo_data_shapley_smoke_test)
    tests.test_in_run_data_shapley_smoke_test(in_run_data_shapley_smoke_test)
    tests.test_random_data_attribution_failure_smoke_test(
        random_data_attribution_failure_smoke_test
    )
    tests.test_label_shuffled_attribution_failure_smoke_test(
        label_shuffled_attribution_failure_smoke_test
    )


## Exercise 6 - notebook contract and runtime

Package the local evidence and measure runtime overhead for full update, exact enumeration, and in-run proxy paths.

<details><summary>Expected output</summary>

```text
All tests in `test_runtime_overhead_smoke_test` passed!
All tests in `test_notebook_contract` passed!
```

</details>

<details><summary>Common bugs</summary>

- Returning tensors instead of lists.
- Omitting the negative controls from the contract.
- Reporting runtime without repeated measurements.

</details>

<details><summary>Solution</summary>

Return a dictionary with exact, Monte Carlo, in-run, both controls, and runtime-overhead metrics.

</details>


In [ ]:
def runtime_overhead_smoke_test(repeats: int = DATA_SHAPLEY_RUNTIME_REPEATS) -> dict:
    """Measure full-update, exact-enumeration, and in-run score overhead."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


def run_smoke_test(cpu: bool = True) -> dict:
    """Package the local Data Shapley evidence."""
    # EXERCISE
    # YOUR CODE HERE
    raise NotImplementedError()
    # END EXERCISE


if MAIN:
    tests.test_runtime_overhead_smoke_test(runtime_overhead_smoke_test)
    tests.test_notebook_contract(run_smoke_test)


## Exercise 7 - CUDA report interpretation

The committed report reruns the finite problem on CUDA, including exact enumeration, an actual optimizer step, in-run autograd scores, controls, and runtime.

<details><summary>Expected output</summary>

```text
All tests in `test_committed_gpu_report_records_exact_proxy_and_controls` passed!
{
  "exact_values": [0.6412, 0.6412, 0.6412, -1.1736],
  "sampled_max_abs_error": 0.0641,
  "gradient_scores": [4.0, 4.0, 4.0, -4.0],
  "random_data_attribution_fails": true,
  "label_shuffled_attribution_fails": true
}
```

</details>

<details><summary>Help - what does this prove?</summary>

It proves the exact/proxy/control pattern on a generated one-step CUDA model organism. It does not prove production data valuation, long-horizon optimizer influence, or real-dataset TracIn behavior.

</details>


In [ ]:
def load_committed_gpu_report() -> dict:
    """Load the committed CUDA report for interpretation inside the notebook."""
    report = json.loads((section_dir / "verification_report.json").read_text())
    return report["metrics"]["gpu_test"]


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    """Run the full CUDA experiment from the section solution module."""
    from chapter16_shapley_attribution_baselines.exercises.part7_data_shapley_in_one_training_run import solutions

    return solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    """Alias used by the repository verification harness."""
    return run_gpu_test(max_vram_gb=max_vram_gb)


if MAIN:
    gpu_report = load_committed_gpu_report()
    tests.test_committed_gpu_report_records_exact_proxy_and_controls()
    summary = {
        "exact_values": gpu_report["exact_values"],
        "sampled_max_abs_error": gpu_report["sampled_max_abs_error"],
        "gradient_scores": gpu_report["gradient_scores"],
        "random_data_attribution_fails": gpu_report["random_data_attribution_fails"],
        "label_shuffled_attribution_fails": gpu_report["label_shuffled_attribution_fails"],
        "peak_vram_gb": gpu_report["peak_vram_gb"],
    }
    print(json.dumps(summary, indent=2))


## Signature Result

<img src="../../instructions/assets/data_shapley_signature_result.svg" width="780">

| Example | Exact Data Shapley | Sampled estimate | In-run score | Interpretation |
|---:|---:|---:|---:|---|
| 0 | `0.6412` | `0.6613` | `4.0` | helpful |
| 1 | `0.6412` | `0.6403` | `4.0` | helpful |
| 2 | `0.6412` | `0.6861` | `4.0` | helpful |
| 3 | `-1.1736` | `-1.2377` | `-4.0` | flipped label, harmful |

<details><summary>What this section shows</summary>

- Exact Data Shapley identifies the flipped-label example as harmful.
- Sampled permutation Data Shapley preserves the harmful-example ranking.
- The one-run gradient-dot proxy correlates with exact values on this toy task.
- Random-data and label-shuffled controls fail, as they should.

</details>

## Limitations

<details><summary>What this section does not show</summary>

- It does not prove production-scale data valuation.
- It does not prove one-run proxies are exact for arbitrary optimizers or long training runs.
- It does not claim TracIn or influence-function parity.
- It does not use private, unsafe, or real user data.

</details>

## Bonus / anomaly hunting

- Increase the number of flipped labels and watch exact values shift.
- Add duplicated helpful examples and inspect how Shapley splits credit.
- Change the learning rate until the first-order proxy breaks.
- Replace squared loss with logistic loss and rerun the exact finite game.
